## 16 - Merge RegBL and Karst

This notebooks' objective is to merge two datasets in their `processed` versions: 

- `11_OFEV_hydrogeo_karst`
- `03_OFS_regbl`

In [ ]:
import pandas as pd
import missingno as msno
import geopandas as gpd
from shapely.geometry import Point
import rasterio
import matplotlib.pyplot as plt
import ezdxf
from rasterio.plot import show
import numpy as np
from rasterio.crs import CRS
import fiona

## Constants

In [ ]:
ROOT = '../../'

## Load the data

In [ ]:
gdb_path = f"{ROOT}data/raw/11_OFEV_hydrogeo_karst/hydrogeo_karst.gdb"

layers = fiona.listlayers(gdb_path)

print(layers)

In [ ]:
# Print the name of the columns in each layer
for layer in layers:
    with fiona.open(gdb_path, layer=layer) as src:
        print(f"Layer: {layer}")
        print("Columns:", src.schema['properties'].keys())
        print()

In [ ]:
df_regbl = pd.read_csv(f'{ROOT}data/processed/dataset-v0.12.csv', index_col=0)

df_regbl.head()

## Convert main DataFrame to GeoDataFrame

In [ ]:
df_regbl = gpd.GeoDataFrame(df_regbl, geometry=gpd.points_from_xy(df_regbl["Coordonnée E"], df_regbl["Coordonnée N"]), crs='EPSG:2056')

## Perform spatial joints on layers

In [ ]:
layers = ['BA', 'UB', 'NK', 'EI', 'IP']

karst_gdf = df_regbl.copy()

for layer in layers:
    try:
        layer_gdf = gpd.read_file(gdb_path, layer=layer)
        if layer_gdf.empty:
            print(f"Skipping {layer} (empty)")
            continue

        # Reproject
        if layer_gdf.crs != df_regbl.crs:
            layer_gdf = layer_gdf.to_crs(df_regbl.crs)

        # Drop index_right from previous join if it exists
        if 'index_right' in karst_gdf.columns:
            karst_gdf = karst_gdf.drop(columns=['index_right'])

        # Perform spatial join
        karst_gdf = gpd.sjoin(karst_gdf, layer_gdf, how="left", predicate="intersects")

    except Exception as e:
        print(f"Could not load or join {layer}: {e}")



In [ ]:
cols = [
    'BA_ID', 'BA_Name', 'BA_Alt_min', 'BA_Alt_moy', 'BA_Alt_max', 'BA_UStruct', 'BA_DD', 'BA_Canton', 'BA_Rem', 'BA_Aqui', 'SHAPE_Length', 'SHAPE_Area', 'BA_Surf',
    'UB_ID', 'UB_Name', 'UB_Type', 'UB_Flux', 'UB_Alt_min', 'UB_Alt_moy', 'UB_Alt_max', 'UB_DD', 'UB_Rem', 'Shape_Length', 'Shape_Area',
    'FID_NK', 'NK_ID', 'NK_Hdyn', 'NK_Type', 'NK_Level', 'MNK_ID', 'Shape_Length', 'Shape_Area',
    'EI_ID', 'EI_Hdyn', 'EI_Type', 'Shape_Length',
    'IP_ID', 'IP_Name', 'IP_Type', 'IP_QClass', 'IP_Reg', 'IP_Exp', 'IP_Prop', 'ID_Group', 'IP_X', 'IP_Y', 'IP_Z', 'IP_Rem'
]

# Add 'Karst - ' to the beginning of each column name in the list
karst_gdf = karst_gdf.rename(columns={col: 'Karst - ' + col for col in cols if col in karst_gdf.columns})

In [ ]:
# Drop the columns that have all NaN values
karst_gdf = karst_gdf.dropna(axis=1, how='all')

In [ ]:
karst_gdf.head()

In [ ]:
karst_gdf.shape

In [ ]:
msno.matrix(karst_gdf, figsize=(20, 10))

## Save the data

In [ ]:
karst_gdf.to_csv(f"{ROOT}data/processed/dataset-v0.13.csv")